# Enhanced Summary Knowledge Tuning - Data Generation

## Overview

This notebook demonstrates how to generate high-quality knowledge tuning datasets using the SDG Hub framework. It creates multiple types of document augmentations and corresponding question-answer pairs that can be used to train or fine-tune language models for enhanced summarization and knowledge extraction capabilities.

## What This Notebook Does

This notebook will:

2. **Generate Four Types of Knowledge Tuning Datasets**:
   - **Extractive Summaries**: Concise summaries that extract key information directly from source documents
   - **Detailed Summaries**: Comprehensive summaries that provide thorough coverage of document content
   - **Key Facts**: Structured fact extraction with corresponding Q&A pairs
   - **Document-Based Q&A**: Question-answer pairs generated directly from document content


4. **Output Structured Training Data**:
   - For each augmentation we save JSONL dataset.
   - You can follow [knowledge_mixing](knowledge_mixing.ipynb) to convert it into training dataset

## Prerequisites

- SDG Hub installed and configured
- Environment variables set up (see [.env.example](.env.example)). Specifically set the model provider, seed data and output path.
- Document pre-processing completed (run [document_pre_processing.ipynb](document_pre_processing.ipynb) first)

```bash 
git clone https://github.com/Red-Hat-AI-Innovation-Team/sdg_hub.git
cd sdg_hub
pip install .[examples]
copy the .env.example to .env and set the model endpoint and generation/mixing parameters
```
**⚠️ If you haven't already, run the document pre-processing notebook to create the seed data.**

## Next Steps

After running this notebook, use [knowledge_mixing](knowledge_mixing.ipynb) to combine and curate the generated datasets for final model training.


In [2]:
# Third Party
from datasets import load_dataset
from dotenv import load_dotenv

# First Party
from sdg_hub import Flow, FlowRegistry
from sdg_hub.core.utils.translation import translate_flow
import os
from pathlib import Path

# Load environment variables from .env file
load_dotenv(override=True)

True

In [6]:
# Load seed data. If one is not provided, create it from the quality benchmark dataset.
seed_data_path = 'data/seed_data.jsonl'

print(f"Loading existing seed data from {seed_data_path}")
quality_corpus = load_dataset("json", data_files=seed_data_path, split="train")
quality_corpus = quality_corpus.to_pandas()

Loading existing seed data from data/seed_data.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

### Run SDG
- This will create knowledge flow from provided yaml file
- We will run this on small dataset for demo purposes
- For large scale generation, please use the python command provided in the next cell
- You can analyze the generated data to ensure the quality is similar to proivded QnA pairs

In [7]:
quality_corpus = quality_corpus.sample(10)

#### Discover the available generation flows

In [22]:
# Auto-discover all available flows (no setup needed!)
FlowRegistry.discover_flows()

# List available flows
flows = FlowRegistry.list_flows()
print(f"Available flows: {flows}")

# You can also search the flows by tag
qa_flows = FlowRegistry.search_flows(tag="question-generation")
print(f"QA flows: {qa_flows}")

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID                  ┃ Name                 ┃ Author               ┃ Tags                 ┃ Description          ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ clean-shadow-397    │ Advanced Japanese    │ SDG Hub Contributors │ question-generation, │ A comprehensive flow │
│                     │ Document Grounded    │                      │ knowledge-extractio… │ that generates       │
│                     │ Question-Answer      │                      │ qa-pairs,            │ high-quality         │
│                     │ Generation Flow for  │                      │ document-processing, │ question-answer      │
│                     │ Knowledge Tuning     │                      │ educational,         │ pairs from Japanese  │
│                     │                      │                      │ multilingual,        │ input documents      │
│                     │                      │                      │ japanese             │ using multiple LLM   │
│                     │                      │                      │                      │ blocks for question  │
│                     │                      │                      │                      │ generation, answer   │
│                     │                      │                      │                      │ synthesis, and       │
│                     │                      │                      │                      │ quality evaluation.  │
│ epic-jade-656       │ Extractive Summary   │ SDG Hub Contributors │ knowledge-tuning,    │ Generate extractive  │
│                     │ Knowledge Tuning     │                      │ document-internaliz… │ summary from the     │
│                     │ Dataset Generation   │                      │ question-generation, │ input document. Each │
│                     │ Flow                 │                      │ knowledge-extractiv… │ document is first    │
│                     │                      │                      │ qa-pairs,            │ converted into list  │
│                     │                      │                      │ extractive-summaries │ of knowledge         │
│                     │                      │                      │                      │ segments for         │
│                     │                      │                      │                      │ creating extractive  │
│                     │                      │                      │                      │ summary and then     │
│                     │                      │                      │                      │ annotated with       │
│                     │                      │                      │                      │ context,             │
│                     │                      │                      │                      │ relationship and     │
│                     │                      │                      │                      │ relevance. This is   │
│                     │                      │                      │                      │ then converted into  │
│                     │                      │                      │                      │ Question-Answer      │
│                     │                      │                      │                      │ pairs.               │
│ epic-jade-656-es    │ Extractive Summary   │ SDG Hub Contributors │ knowledge-tuning,    │ Generate extractive  │
│                     │ Knowledge Tuning     │                      │ document-internaliz… │ summary from the     │
│                     │ Dataset Generation   │                      │ question-generation, │ input document in    │
│                     │ Flow (Spanish)       │                      │ knowledge-extractiv… │ Spanish. Each        │
│                     │                      │          

Available flows: [{'id': 'clean-shadow-397', 'name': 'Advanced Japanese Document Grounded Question-Answer Generation Flow for Knowledge Tuning'}, {'id': 'stellar-peak-605-es', 'name': 'Document Based Knowledge Tuning Dataset Generation Flow (Spanish)'}, {'id': 'heavy-heart-77-es', 'name': 'Key Facts Knowledge Tuning Dataset Generation Flow (Spanish)'}, {'id': 'mild-thunder-748-es', 'name': 'Detailed Summary Knowledge Tuning Dataset Generation Flow (Spanish)'}, {'id': 'epic-jade-656-es', 'name': 'Extractive Summary Knowledge Tuning Dataset Generation Flow (Spanish)'}, {'id': 'stellar-peak-605', 'name': 'Document Based Knowledge Tuning Dataset Generation Flow'}, {'id': 'heavy-heart-77', 'name': 'Key Facts Knowledge Tuning Dataset Generation Flow'}, {'id': 'mild-thunder-748', 'name': 'Detailed Summary Knowledge Tuning Dataset Generation Flow'}, {'id': 'epic-jade-656', 'name': 'Extractive Summary Knowledge Tuning Dataset Generation Flow'}, {'id': 'green-clay-812', 'name': 'Structured Text 

#### Multilingual Support (Optional)

If `SDG_LANG` and `SDG_LANG_CODE` are set in your `.env` file, the notebook will
use translated flow variants. If translated flows are not yet available in the
FlowRegistry, they will be created automatically using `translate_flow()`.

Delete `SDG_LANG` from your `.env` to use the default English flows.

In [8]:
# Read multilingual settings from .env
sdg_lang = os.getenv("SDG_LANG", "").strip()
sdg_lang_code = os.getenv("SDG_LANG_CODE", "").strip()

if sdg_lang and not sdg_lang_code:
    raise ValueError("SDG_LANG is set but SDG_LANG_CODE is missing. Both are required.")
if sdg_lang_code and not sdg_lang:
    raise ValueError("SDG_LANG_CODE is set but SDG_LANG is missing. Both are required.")

# If a custom translated flows directory exists, register it for discovery
translated_flows_dir = os.getenv("TRANSLATED_FLOWS_DIR", "").strip()
if translated_flows_dir and os.path.isdir(translated_flows_dir):
    FlowRegistry.register_search_path(translated_flows_dir)
    FlowRegistry._discover_flows(force_refresh=True)


def ensure_translated_flow(base_flow_name: str) -> str:
    """Return the flow name to use, translating on-demand if needed.

    When SDG_LANG is unset the base (English) name is returned unchanged.
    Otherwise checks FlowRegistry for a pre-existing translated variant
    and falls back to running translate_flow() to create one.
    """
    if not sdg_lang:
        return base_flow_name

    translated_name = f"{base_flow_name} ({sdg_lang})"
    if FlowRegistry.get_flow_path(translated_name) is not None:
        print(f"  ✓ Found: {translated_name}")
        return translated_name

    # Translate on-demand
    print(f"  ⟳ Translating: {base_flow_name} → {sdg_lang}...")
    # Compute per-flow output subdir to avoid flow.yaml collisions
    source_path = FlowRegistry.get_flow_path(base_flow_name)
    flow_subdir = Path(source_path).parent.name
    output_dir = None
    if translated_flows_dir:
        output_dir = str(Path(translated_flows_dir) / f"{flow_subdir}_{sdg_lang_code}")

    translate_flow(
        flow=base_flow_name,
        lang=sdg_lang,
        lang_code=sdg_lang_code,
        translator_model=os.getenv("TRANSLATOR_MODEL", "openai/gpt-4o"),
        verifier_model=os.getenv(
            "VERIFIER_MODEL", os.getenv("TRANSLATOR_MODEL", "openai/gpt-4o")
        ),
        output_dir=output_dir,
        translator_api_key=os.getenv("TRANSLATOR_API_KEY"),
        translator_api_base=os.getenv("TRANSLATOR_API_BASE"),
        verifier_api_key=os.getenv("VERIFIER_API_KEY"),
        verifier_api_base=os.getenv("VERIFIER_API_BASE"),
        register=True,
    )
    return translated_name


# Resolve all flow names (translating if needed)
BASE_FLOW_NAMES = [
    "Extractive Summary Knowledge Tuning Dataset Generation Flow",
    "Detailed Summary Knowledge Tuning Dataset Generation Flow",
    "Key Facts Knowledge Tuning Dataset Generation Flow",
    "Document Based Knowledge Tuning Dataset Generation Flow",
]

if sdg_lang:
    print(f"Multilingual mode: {sdg_lang} ({sdg_lang_code})\n")
else:
    print("Language: English (default)\n")

flow_names = {}
for name in BASE_FLOW_NAMES:
    flow_names[name] = ensure_translated_flow(name)

print(f"\nFlows to use: {list(flow_names.values())}")

Language: English (default)


Flows to use: ['Extractive Summary Knowledge Tuning Dataset Generation Flow', 'Detailed Summary Knowledge Tuning Dataset Generation Flow', 'Key Facts Knowledge Tuning Dataset Generation Flow', 'Document Based Knowledge Tuning Dataset Generation Flow']


In [9]:
# Get runtime parameters
enable_reasoning = os.getenv("ENABLE_REASONING", "false").lower() in (
    "1",
    "true",
    "yes",
)
number_of_summaries = int(os.getenv("NUMBER_OF_SUMMARIES", "50"))
max_concurrency = int(os.getenv("MAX_CONCURRENCY", "8"))
save_data_path = os.getenv("OUTPUT_DATA_FOLDER", "")
generate_cpt = os.getenv("GENERATE_CPT", "false").lower() in (
    "1",
    "true",
    "yes",
)

In [11]:
flow_name = 'Extractive Summary Knowledge Tuning Dataset Generation Flow'
flow_path = FlowRegistry.get_flow_path(flow_name)
flow_path

[18:14:03] INFO     Discovered 12 flows                                                             ]8;id=378167;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/registry.py\registry.py]8;;\:]8;id=586454;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/registry.py#126\126]8;;\

'/workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/flows/knowledge_infusion/enhanced_multi_summary_qa/extractive_summary/flow.yaml'

In [12]:
!cat '/workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/flows/knowledge_infusion/enhanced_multi_summary_qa/extractive_summary/flow.yaml'

cat: /workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/sdg_hub/flows/knowledge_infusion/enhanced_multi_summary_qa/extractive_summary/flow.yaml: No such file or directory


In [13]:
import nest_asyncio; nest_asyncio.apply()

In [14]:
flow = Flow.from_yaml(flow_path)

# Set model configuration
# flow = set_model_config(flow)
flow.set_model_config(
    model="hosted_vllm/glm-4.7-fp8",
    api_base="http://localhost:8100/v1",
    timeout=1200.0,
)


[18:14:43] INFO     Loading flow from:                                                          ]8;id=908602;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/serialization.py\serialization.py]8;;\:]8;id=240753;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/serialization.py#60\60]8;;\
                    /workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-package                    
                    s/sdg_hub/flows/knowledge_infusion/enhanced_multi_summary_qa/extractive_sum                    
                    mary/flow.yaml                                                                                 

[18:14:43] INFO     Auto-detected 4 LLM blocks for configuration: ['answer_generation',         ]8;id=592590;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=747176;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py#242\242]8;;\
                    'eval_faithful_llm_chat', 'gen_extractive_summary', 'question_generation']                     

           INFO     Successfully configured 4 LLM blocks with: model:                           ]8;id=742021;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=405385;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py#300\300]8;;\
                    'hosted_vllm/glm-4.7-fp8', api_base: 'http://localhost:8100/v1', timeout:                      
                    '1200.0'                                                                                       

           INFO     Configured blocks: ['answer_generation', 'eval_faithful_llm_chat',          ]8;id=807419;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=244617;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py#303\303]8;;\
                    'gen_extractive_summary', 'question_generation']                                               

In [16]:
runtime_params = {
    "question_generation": {"max_tokens": 32768},
    "gen_extractive_summary": {"n": 50, "max_tokens": 32768},
}

extractive_summary_generated_data = flow.generate(
    quality_corpus.sample(1), runtime_params=runtime_params, max_concurrency=max_concurrency
)

[18:16:36] INFO     Using max_concurrency=8 for LLM requests                                       ]8;id=946688;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=281220;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py#419\419]8;;\

           INFO     Starting flow 'Extractive Summary Knowledge Tuning Dataset Generation Flow'    ]8;id=761985;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=609405;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py#448\448]8;;\
                    v2.0.0 with 1 samples across 20 blocks (max_concurrency=8)                                     

           INFO     Executing block 1/20: duplicate_document_col (DuplicateColumnsBlock)           ]8;id=493816;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=368525;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py#220\220]8;;\

╭──────────────────────────────────────────── duplicate_document_col ─────────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: DuplicateColumnsBlock                                                                               │
│ Input Rows: 1                                                                                                   │
│ Input Columns: 7                                                                                                │
│ Column Names: document, document_outline, icl_document, icl_query_1, icl_query_2, icl_query_3, domain           │
│ Expected Output Columns: base_document                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── duplicate_document_col - Complete ───────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 1 → 1                                                                                                     │
│ Columns: 7 → 8                                                                                                  │
│ 🟢 Added: base_document                                                                                         │
│ 📋 Final Columns: base_document, document, document_outline, domain, icl_document, icl_query_1, icl_query_2,    │
│ icl_query_3                                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── eval_faithful_filter ──────────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: ColumnValueFilterBlock                                                                              │
│ Input Rows: 359                                                                                                 │
│ Input Columns: 25                                                                                               │
│ Column Names: raw_document, document_outline, icl_document, icl_query_1, icl_query_2, icl_query_3, domain,      │
│ base_document, extractive_summary_prompt, raw_summary, extract_extractive_summary_content, document,            │
│ question_generation_prompt, question_list, extract_questions_content, question, answer_generation_prompt,       │
│ response_dict, extract_answers_content, response, eval_faithful_prompt, eval_faithful_response_dict,            │
│ extract_eval_faithful_content, faithfulness_explanation, faithfulness_judgement                                 │
│ Expected Output Columns: None specified                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── eval_faithful_filter - Complete ────────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 359 → 359                                                                                                 │
│ Columns: 25 → 25                                                                                                │
│ 📋 Final Columns: answer_generation_prompt, base_document, document, document_outline, domain,                  │
│ eval_faithful_prompt, eval_faithful_response_dict, extract_answers_content, extract_eval_faithful_content,      │
│ extract_extractive_summary_content, extract_questions_content, extractive_summary_prompt,                       │
│ faithfulness_explanation, faithfulness_judgement, icl_document, icl_query_1, icl_query_2, icl_query_3,          │
│ question, question_generation_prompt, question_list, raw_document, raw_summary, response, response_dict         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Block 'eval_faithful_filter' completed successfully: 359 samples, 25 columns   ]8;id=596185;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=41912;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py#266\266]8;;\

╭──────────────────── Extractive Summary Knowledge Tuning Dataset Generation Flow - Complete ─────────────────────╮
│                                        Flow Execution Summary                                                   │
│ ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓           │
│ ┃ Block Name           ┃ Type            ┃   Duration ┃     Rows     ┃     Columns     ┃   Status   ┃           │
│ ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩           │
│ │ duplicate_document_… │ DuplicateColum… │      0.01s │    1 → 1     │       +1        │     ✓      │           │
│ │ extractive_summary_… │ PromptBuilderB… │      0.01s │    1 → 1     │       +1        │     ✓      │           │
│ │ gen_extractive_summ… │ LLMChatBlock    │    336.20s │    1 → 1     │       +1        │     ✓      │           │
│ │ extract_extractive_… │ LLMResponseExt… │      0.01s │    1 → 50    │       +1        │     ✓      │           │
│ │ parse_extractive_su… │ TagParserBlock  │      0.01s │   50 → 50    │       +1        │     ✓      │           │
│ │ rename_to_raw_docum… │ RenameColumnsB… │      0.00s │   50 → 50    │      +1/-1      │     ✓      │           │
│ │ rename_to_document_… │ RenameColumnsB… │      0.00s │   50 → 50    │      +1/-1      │     ✓      │           │
│ │ question_generation… │ PromptBuilderB… │      0.01s │   50 → 50    │       +1        │     ✓      │           │
│ │ question_generation  │ LLMChatBlock    │    426.73s │   50 → 50    │       +1        │     ✓      │           │
│ │ extract_questions    │ LLMResponseExt… │      0.01s │   50 → 50    │       +1        │     ✓      │           │
│ │ parse_question_list  │ TagParserBlock  │      0.03s │   50 → 359   │       +1        │     ✓      │           │
│ │ answer_generation_p… │ PromptBuilderB… │      0.10s │  359 → 359   │       +1        │     ✓      │           │
│ │ answer_generation    │ LLMChatBlock    │    857.88s │  359 → 359   │       +1        │     ✓      │           │
│ │ extract_answers      │ LLMResponseExt… │      0.06s │  359 → 359   │       +1        │     ✓      │           │
│ │ parse_response_dict  │ TagParserBlock  │      0.06s │  359 → 359   │       +1        │     ✓      │           │
│ │ eval_faithful_prompt │ PromptBuilderB… │      0.07s │  359 → 359   │       +1        │     ✓      │           │
│ │ eval_faithful_llm_c… │ LLMChatBlock    │   2059.45s │  359 → 359   │       +1        │     ✓      │           │
│ │ extract_eval_faithf… │ LLMResponseExt… │      0.05s │  359 → 359   │       +1        │     ✓      │           │
│ │ parse_eval_faithful  │ TagParserBlock  │      0.06s │  359 → 359   │       +2        │     ✓      │           │
│ │ eval_faithful_filter │ ColumnValueFil… │      0.00s │  359 → 359   │        —        │     ✓      │           │
│ ├──────────────────────┼─────────────────┼────────────┼──────────────┼─────────────────┼────────────┤           │
│ │ TOTAL                │ 20 blocks       │   3680.75s │  359 final   │    25 final     │   20/20    │           │
│ └──────────────────────┴─────────────────┴────────────┴──────────────┴─────────────────┴────────────┘           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Flow 'Extractive Summary Knowledge Tuning Dataset Generation Flow' completed   ]8;id=717759;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=182098;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py#548\548]8;;\
                    successfully: 359 final samples, 25 final columns                                              

In [17]:
print(f"✓ Extractive summary: {len(extractive_summary_generated_data)} records")
print(f"✓ Columns: {list(extractive_summary_generated_data.columns.tolist())}")

✓ Extractive summary: 359 records
✓ Columns: ['raw_document', 'document_outline', 'icl_document', 'icl_query_1', 'icl_query_2', 'icl_query_3', 'domain', 'base_document', 'extractive_summary_prompt', 'raw_summary', 'extract_extractive_summary_content', 'document', 'question_generation_prompt', 'question_list', 'extract_questions_content', 'question', 'answer_generation_prompt', 'response_dict', 'extract_answers_content', 'response', 'eval_faithful_prompt', 'eval_faithful_response_dict', 'extract_eval_faithful_content', 'faithfulness_explanation', 'faithfulness_judgement']


In [ ]:
os.makedirs(os.path.join(save_data_path, "extractive_summary"), exist_ok=True)

extractive_summary_generated_data.to_json(
    os.path.join(save_data_path, "extractive_summary", "gen.jsonl"),
    orient="records",
    lines=True,
)

In [25]:
quality_corpus.shape

(10, 7)

In [26]:
extractive_summary_generated_data.shape

(359, 25)

In [27]:
extractive_summary_generated_data.columns.tolist()

['raw_document',
 'document_outline',
 'icl_document',
 'icl_query_1',
 'icl_query_2',
 'icl_query_3',
 'domain',
 'base_document',
 'extractive_summary_prompt',
 'raw_summary',
 'extract_extractive_summary_content',
 'document',
 'question_generation_prompt',
 'question_list',
 'extract_questions_content',
 'question',
 'answer_generation_prompt',
 'response_dict',
 'extract_answers_content',
 'response',
 'eval_faithful_prompt',
 'eval_faithful_response_dict',
 'extract_eval_faithful_content',
 'faithfulness_explanation',
 'faithfulness_judgement']

In [ ]:
x = extractive_summary_generated_data.iloc[0]

print(x['question'])
print(x['response'])


Based on empirical evidence, which training strategy is advocated over sequential phased training?


KeyError: 'answer'

### Detailed Summary

In [19]:
# Generate similar data for Detailed Summary
flow_name = flow_names["Detailed Summary Knowledge Tuning Dataset Generation Flow"]
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
# flow = set_model_config(flow)
flow.set_model_config(
    model="hosted_vllm/glm-4.7-fp8",
    api_base="http://10.241.128.25:8100/v1",
    timeout=600.0,
)

[19:32:51] INFO     Loading flow from:                                                          ]8;id=254784;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/serialization.py\serialization.py]8;;\:]8;id=887584;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/serialization.py#60\60]8;;\
                    /workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-package                    
                    s/sdg_hub/flows/knowledge_infusion/enhanced_multi_summary_qa/detailed_summa                    
                    ry/flow.yaml                                                                                   

[19:32:51] INFO     Auto-detected 4 LLM blocks for configuration: ['answer_generation',         ]8;id=596354;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=66678;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py#242\242]8;;\
                    'eval_faithful_llm_chat', 'gen_detailed_summary', 'question_generation']                       

           INFO     Successfully configured 4 LLM blocks with: model:                           ]8;id=273846;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=77118;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py#300\300]8;;\
                    'hosted_vllm/glm-4.7-fp8', api_base: 'http://10.241.128.25:8100/v1',                           
                    timeout: '600.0'                                                                               

           INFO     Configured blocks: ['answer_generation', 'eval_faithful_llm_chat',          ]8;id=914762;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=317843;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/model_config.py#303\303]8;;\
                    'gen_detailed_summary', 'question_generation']                                                 

In [20]:
detailed_summary_generated_data = flow.generate(
    quality_corpus.sample(1), runtime_params=runtime_params, max_concurrency=max_concurrency
)

[19:33:20] INFO     Using max_concurrency=8 for LLM requests                                       ]8;id=1996;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=692587;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py#419\419]8;;\

           INFO     Starting flow 'Detailed Summary Knowledge Tuning Dataset Generation Flow'      ]8;id=305256;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=289831;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py#448\448]8;;\
                    v2.0.0 with 1 samples across 20 blocks (max_concurrency=8)                                     

           INFO     Executing block 1/20: duplicate_document_col (DuplicateColumnsBlock)           ]8;id=815141;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=352979;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/flow/execution.py#220\220]8;;\

╭──────────────────────────────────────────── duplicate_document_col ─────────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: DuplicateColumnsBlock                                                                               │
│ Input Rows: 1                                                                                                   │
│ Input Columns: 7                                                                                                │
│ Column Names: document, document_outline, icl_document, icl_query_1, icl_query_2, icl_query_3, domain           │
│ Expected Output Columns: base_document                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── duplicate_document_col - Complete ───────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 1 → 1                                                                                                     │
│ Columns: 7 → 8                                                                                                  │
│ 🟢 Added: base_document                                                                                         │
│ 📋 Final Columns: base_document, document, document_outline, domain, icl_document, icl_query_1, icl_query_2,    │
│ icl_query_3                                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[19:33:20] INFO     Starting async generation for 1 samples (max_concurrency=8)               ]8;id=274644;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/blocks/llm/llm_chat_block.py\llm_chat_block.py]8;;\:]8;id=277937;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/blocks/llm/llm_chat_block.py#215\215]8;;\

           WARNING  max_concurrency (8) is less than n (50). Consider increasing              ]8;id=967726;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/blocks/llm/llm_chat_block.py\llm_chat_block.py]8;;\:]8;id=234987;file:///workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/sdg_hub/core/blocks/llm/llm_chat_block.py#487\487]8;;\
                    max_concurrency for optimal performance.                                                       

gen_detailed_summary:   0%|                                                                                                                                            | 0/1 [01:05<?, ?req/s]


╭────────────────────── Detailed Summary Knowledge Tuning Dataset Generation Flow - Failed ───────────────────────╮
│                                        Flow Execution Summary                                                   │
│ ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓           │
│ ┃ Block Name           ┃ Type            ┃   Duration ┃     Rows     ┃     Columns     ┃   Status   ┃           │
│ ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩           │
│ │ duplicate_document_… │ DuplicateColum… │      0.00s │    1 → 1     │       +1        │     ✓      │           │
│ │ detailed_summary_pr… │ PromptBuilderB… │      0.00s │    1 → 1     │       +1        │     ✓      │           │
│ ├──────────────────────┼─────────────────┼────────────┼──────────────┼─────────────────┼────────────┤           │
│ │ TOTAL                │ 2 blocks        │      0.01s │   0 final    │     0 final     │    2/2     │           │
│ └──────────────────────┴─────────────────┴────────────┴──────────────┴─────────────────┴────────────┘           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

KeyboardInterrupt: 

In [ ]:
print(f"✓ Detailed summary: {len(detailed_summary_generated_data)} records")
print(f"✓ Columns: {list(detailed_summary_generated_data.columns.tolist())}")

In [ ]:
os.makedirs(os.path.join(save_data_path, "detailed_summary"), exist_ok=True)
detailed_summary_generated_data.to_json(
    os.path.join(save_data_path, "detailed_summary", "gen.jsonl"),
    orient="records",
    lines=True,
)


In [ ]:
# Generate similar data for key facts
flow_name = flow_names["Key Facts Knowledge Tuning Dataset Generation Flow"]
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
# flow = set_model_config(flow)
flow.set_model_config(
    model="hosted_vllm/glm-4.7-fp8",
    api_base="http://10.241.128.25:8100/v1",
    timeout=600.0,
)


# Remove all blocks after document augmentation
if generate_cpt:
    kept_blocks = []
    for block in flow.blocks:
        if block.block_name == "parse_atomic_facts_to_individual_facts":
            break
        kept_blocks.append(block)
    flow.blocks = kept_blocks

runtime_params = {}
if enable_reasoning:
    # Increase max tokens for Question Generation to accommodate reasoning content
    runtime_params = {
        "generate_key_fact_qa": {"max_tokens": 6000},
    }

# Generate data for key facts summary
key_facts_generated_data = flow.generate(
    quality_corpus, runtime_params=runtime_params, max_concurrency=max_concurrency
)

os.makedirs(os.path.join(save_data_path, "key_facts_to_qa"), exist_ok=True)

key_facts_generated_data.to_json(
    os.path.join(save_data_path, "key_facts_to_qa" if not generate_cpt else "key_facts_cpt", "gen.jsonl"),
    orient="records",
    lines=True,
)

print(f"✓ Key facts: {len(key_facts_generated_data)} records")

print(f"✓ Columns: {list(key_facts_generated_data.columns.tolist())}")

In [ ]:
flow_name = flow_names["Document Based Knowledge Tuning Dataset Generation Flow"]
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
# flow = set_model_config(flow)
flow.set_model_config(
    model="hosted_vllm/glm-4.7-fp8",
    api_base="http://10.241.128.25:8100/v1",
    timeout=600.0,
)
runtime_params = {}
if enable_reasoning:
    # Increase max tokens to accommodate reasoning content
    runtime_params = {
        "question_generation": {"max_tokens": 2048},
    }

if generate_cpt:
    # Skip generation if CPT is enabled. We simply train on the original document
    document_based_generated_data = quality_corpus[["document", "document_outline", "domain"]]
else:
    document_based_generated_data = flow.generate(
        quality_corpus, runtime_params=runtime_params, max_concurrency=max_concurrency
    )

os.makedirs(os.path.join(save_data_path, "document_based_qa"), exist_ok=True)

document_based_generated_data.to_json(
    os.path.join(save_data_path, "document_based_qa" if not generate_cpt else "document_based_cpt", "gen.jsonl"),
    orient="records",
    lines=True,
)

print(f"✓ Document based: {len(document_based_generated_data)} records")

print(f"✓ Columns: {list(document_based_generated_data.columns.tolist())}")

🎉 You now have all three four of document augmentations (detailed summaries, extractive summaries, key facts and document based) along with their corresponding QA pairs.

✅ Next steps:
   - Combine and curate these datasets to prepare your final training data.
   - For detailed guidance on post-processing, mixing, and formatting the data for model training (including conversion to messages format), please refer to [knowledge_mixing.ipynb](knowledge_mixing.ipynb).